# OpsiGen Pipeline Example

This notebook demonstrates the refactored OpsiGen workflow using Python imports instead of shell commands. It covers configuration loading, input validation, preprocessing, training, batch prediction, and output inspection.

![OpsiGen overview](Images/main_figure.png)


## 1. Imports and paths

Run this notebook from the repository root so relative paths in `configs/*.json` resolve as shown. If you installed the package with `pip install -e .`, the imports below work from any notebook location.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from opsigen.config import PredictionConfig, TrainingConfig
from opsigen.preprocessing import preprocess_opsins
from opsigen.prediction import run_predictions
from opsigen.training import train_model

REPO_ROOT = Path.cwd()
print(REPO_ROOT)


## 2. Load prediction configuration

The prediction config defines the model, normalization arrays, batch input records, preprocessing settings, and output directory.


In [ ]:
prediction_config = PredictionConfig.from_file(REPO_ROOT / "configs" / "predict.example.json")

print("Model:", prediction_config.model_path)
print("Output directory:", prediction_config.output_dir)
print("Records:")
for record in prediction_config.records:
    print(" -", record.id, record.fasta_path, record.pdb_path)


## 3. Validate expected inputs

This is a lightweight check before launching MAFFT, feature generation, or Torch inference.


In [ ]:
required_paths = [
    prediction_config.model_path,
    prediction_config.means_path,
    prediction_config.stds_path,
]
if prediction_config.preprocessing is not None:
    required_paths.extend([
        prediction_config.preprocessing.reference_alignment,
        prediction_config.preprocessing.feature_maker_binary,
        prediction_config.preprocessing.amino_mapping_path,
    ])
for record in prediction_config.records:
    required_paths.extend(path for path in [record.fasta_path, record.pdb_path, record.features_path, record.dists_path] if path)

validation = pd.DataFrame({
    "path": [str(path) for path in required_paths],
    "exists": [path.exists() for path in required_paths],
})
validation


## 4. Run preprocessing

Preprocessing aligns each FASTA to the reference alignment, cuts the matching residues from the PDB, runs the native feature generator, appends amino-acid descriptors, and writes graph distance matrices.


In [ ]:
RUN_PREPROCESSING = False

if RUN_PREPROCESSING:
    preprocessed = preprocess_opsins(prediction_config.records, prediction_config.preprocessing)
    pd.DataFrame([record.__dict__ for record in preprocessed])
else:
    print("Set RUN_PREPROCESSING = True after MAFFT and the native feature generator are available.")


## 5. Load training configuration

Training is configured separately from prediction. The example config points to graph directories that you provide when training a new model.


In [ ]:
training_config = TrainingConfig.from_file(REPO_ROOT / "configs" / "train.example.json")

print("Training graphs:", training_config.data.graph_features_path)
print("Distance graphs:", training_config.data.graph_dists_path)
print("Model class:", training_config.model.name)
print("Epochs:", training_config.fit.epochs)
print("Run output:", training_config.output_dir)


## 6. Train a model

Training writes metrics, checkpoints, normalization arrays, and metadata to the configured run directory. Leave the toggle off while exploring the notebook, then enable it once graph training data and Torch dependencies are ready.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    training_result = train_model(training_config)
    print(training_result)
else:
    print("Set RUN_TRAINING = True to launch a configured training run.")


## 7. Run batch prediction

Batch prediction uses the same config object. If records do not already include `features_path` and `dists_path`, the prediction pipeline runs preprocessing first.


In [ ]:
RUN_PREDICTION = False

if RUN_PREDICTION:
    prediction_result = run_predictions(prediction_config)
    predictions = pd.read_csv(prediction_result.predictions_path)
    predictions
else:
    print("Set RUN_PREDICTION = True after preprocessing dependencies and Torch are available.")


## 8. Inspect outputs

Prediction outputs are a CSV file plus run metadata. Training outputs include metrics JSONL, final and best checkpoints, normalization arrays, and metadata.


In [ ]:
prediction_csv = prediction_config.output_dir / "predictions.csv"
metadata_json = prediction_config.output_dir / "metadata.json"

if prediction_csv.exists():
    display(pd.read_csv(prediction_csv))
else:
    print("No prediction CSV found yet:", prediction_csv)

if metadata_json.exists():
    metadata = json.loads(metadata_json.read_text())
    print(json.dumps(metadata, indent=2)[:2000])
else:
    print("No prediction metadata found yet:", metadata_json)
